# News Scraper WAFA / PNN / Jerusalem Post / Israel National News

Scrapes English-language news articles from four outlets: [WAFA](https://english.wafa.ps), [PNN](https://english.pnn.ps), [The Jerusalem Post](https://www.jpost.com), and [Israel National News](https://www.israelnationalnews.com), filtered to Gaza/Israel-Palestine coverage published on or after **7 Oct 2023**. Each site gets its own section below since page structure and crawl strategy differ per site, but configuration, keyword matching, and resume/save logic are shared where possible. See `README.md` for setup, usage, and notes on scope.

## 1. Shared Setup

Imports, keyword list, and helper functions used by every site section below.

### Imports

In [ ]:
import os
import re
import gzip
import json
import time
import random
import requests
import pandas as pd
from bs4 import BeautifulSoup
from datetime import datetime
from tqdm import tqdm
from urllib.parse import quote
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as ec
from selenium.common.exceptions import TimeoutException

### Global configuration

Shared across all four scrapers. Site-specific settings (base URLs, ID ranges, output filenames) live in each site's own section.

In [ ]:
headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}

start_date = datetime(2023, 10, 7)  # only keep articles published on/after this date

request_delay = 0.3

# Combined keyword list used to tag/filter articles across all four sources
keywords = [
    "gaza",
    "israel",
    "israeli",
    "palestine",
    "palestinian",
    "hamas",
    "west bank",
    "jerusalem",
    "rafah",
    "ceasefire",
    "occupation",
    "settlers",
    "airstrike",
    "bombing",
    "hostage",
    "war",
    "idf",
    "civilian",
    "hospital",
    "humanitarian",]

### Shared helper functions

In [ ]:
def clean_text(text):
    # Collapse whitespace/line breaks into single spaces
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def get_html(url, timeout=30):
    # Fetch a URL and return its HTML text, or None on any error/non-200 response
    try:
        response = requests.get(url, headers=headers, timeout=timeout)
        if response.status_code != 200:
            return None
        return response.text
    except Exception:
        return None


def find_matching_keywords(text, keyword_list=keywords):
    # Return the sorted list of keywords found in text (case-insensitive)
    text_lower = text.lower()
    matches = [kw for kw in keyword_list if kw.lower() in text_lower]
    return sorted(set(matches))


def load_saved_keys(output_file, key_col):
    # Resume support: load the set of already-saved key values (URL or ID) from an existing CSV
    if os.path.exists(output_file):
        existing = pd.read_csv(output_file, dtype=str)
        saved = set(existing[key_col].dropna())
        print(f"Existing saved rows in {output_file}: {len(existing)}")
        return saved
    print(f"No existing file at {output_file}. Starting from zero.")
    return set()


def append_row(row_dict, output_file):
    # Append a single scraped row to its output CSV, writing the header only if the file is new
    pd.DataFrame([row_dict]).to_csv(
        output_file,
        mode="a",
        header=not os.path.exists(output_file),
        index=False,
        encoding="utf-8-sig",)

---
## 2. WAFA

[english.wafa.ps](https://english.wafa.ps) publishes articles at sequential numeric URLs (`/Pages/Details/<id>`), so this section iterates over an ID range directly.

### Configuration

In [ ]:
wafa_base_url = "https://english.wafa.ps/Pages/Details/"

wafa_start_id = 137957   # first article after 7 Oct 2023
wafa_end_id = 172554     # last article checked (11 Jul 2026)

wafa_output_file = "wafa_raw_articles.csv"

### Scrape a single article

In [ ]:
def scrape_wafa_article(article_id):
    url = wafa_base_url + str(article_id)
    html = get_html(url)
    if html is None:
        return None
    soup = BeautifulSoup(html, "html.parser")

    # Title
    title_tag = soup.find("title")
    if not title_tag:
        return None
    title = clean_text(title_tag.get_text()).replace(" - WAFA Agency", "")

    # This pulls the first
    # "Month DD, YYYY"-shaped string out of the raw page text
    page_text = soup.get_text(" ", strip=True)
    date_match = re.search(r"[A-Z][a-z]+ \d{1,2}, \d{4}", page_text)
    date = None
    if date_match:
        try:
            date = datetime.strptime(date_match.group(), "%B %d, %Y")
        except ValueError:
            date = None

    if date is None or date < start_date:
        return None

    # Body: concatenate <p> tags, dropping short/boilerplate paragraphs
    paragraphs = []
    for p in soup.find_all("p"):
        txt = clean_text(p.get_text(" ", strip=True))
        if len(txt) > 50 and "All Rights Reserved" not in txt:
            paragraphs.append(txt)
    article = clean_text(" ".join(paragraphs))

    if len(article) < 200:
        return None

    matched_keywords = find_matching_keywords(article)

    return {
        "article_id": article_id,
        "url": url,
        "source": "WAFA English",
        "title": title,
        "date": date.strftime("%d-%m-%Y"),
        "content": article,
        "keywords": ", ".join(matched_keywords),}

### Run the WAFA scraper

Resumable: re-running skips article IDs already saved in `wafa_raw_articles.csv`.

In [ ]:
wafa_saved_ids = load_saved_keys(wafa_output_file, key_col="article_id")

for article_id in tqdm(range(wafa_start_id, wafa_end_id + 1)):
    if str(article_id) in wafa_saved_ids:
        continue

    article = scrape_wafa_article(article_id)
    if article is not None:
        append_row(article, wafa_output_file)
        wafa_saved_ids.add(str(article_id))

    time.sleep(request_delay)

print("WAFA scraping complete.")

---
## 3. PNN

[english.pnn.ps](https://english.pnn.ps) also uses sequential numeric article IDs (`/news/<id>`), so this follows the same ID-iteration pattern as WAFA, with PNN-specific parsing (a `Posted On:` date field, a `news-content` body container, and PNN-specific boilerplate to strip).

### Configuration

In [ ]:
pnn_base_url = "https://english.pnn.ps/news/"

pnn_start_id = 46374
pnn_end_id = 48231

pnn_output_file = "pnn_raw_articles.csv"

### Scrape a single article

In [ ]:
# Paragraph-level boilerplate to drop from PNN articles
pnn_unwanted_exact = {
    "contact us",
    "follow us",
    "watch us",
    "latest news",
    "share this news !",
    "share this news!",
    "share this news",}

pnn_unwanted_patterns = [
    r"^all rights reserved",
    r"^designed and developed by",
    r"^copyright",
    r"^©",]


def scrape_pnn_article(article_id):
    url = f"{pnn_base_url}{article_id}"
    html = get_html(url)
    if html is None:
        return None
    soup = BeautifulSoup(html, "html.parser")

    # Title
    title = None
    if soup.title:
        title = soup.title.get_text(" ", strip=True)
        title = re.sub(r"\s*\|\s*PNN.*$", "", title).strip()

    # Date — PNN embeds it as plain text near "Posted On:"
    page_text = soup.get_text(" ", strip=True)
    date_match = re.search(r"Posted On:\s*(\d{2}-\d{2}-\d{4})", page_text)
    date = date_match.group(1) if date_match else None

    # Body
    news_content = soup.find("div", class_="news-content")
    if not news_content:
        return None

    # Skip the first paragraph (dateline byline, e.g. "Bethlehem / Battir /PNN/ ...")
    all_paragraphs = news_content.find_all("p")
    paragraphs = []
    for p in all_paragraphs[1:]:
        text = p.get_text(" ", strip=True)
        if not text:
            continue
        text_lower = text.lower()
        if text_lower in pnn_unwanted_exact:
            continue
        if any(re.search(pat, text_lower) for pat in pnn_unwanted_patterns):
            continue
        paragraphs.append(text)

    if not paragraphs:
        return None
    content = "\n\n".join(paragraphs)

    matched_keywords = find_matching_keywords(content)
    if not matched_keywords:
        return None  # only keep articles matching the conflict keyword list

    return {
        "article_id": article_id,
        "url": url,
        "source": "PNN English",
        "title": title,
        "date": date,
        "content": content,
        "keywords": ", ".join(matched_keywords),}

### Run the PNN scraper

In [ ]:
pnn_saved_urls = load_saved_keys(pnn_output_file, key_col="url")

for article_id in tqdm(range(pnn_start_id, pnn_end_id + 1)):
    url = f"{pnn_base_url}{article_id}"
    if url in pnn_saved_urls:
        continue

    article = scrape_pnn_article(article_id)
    if article is not None:
        append_row(article, pnn_output_file)
        pnn_saved_urls.add(url)

    time.sleep(request_delay)

print("PNN scraping complete.")

---
## 4. The Jerusalem Post

JPost doesn't use sequential IDs, so this section works differently: first discover article URLs from the site's XML sitemap, filter to relevant sections, then scrape each article's `NewsArticle` JSON-LD block (which conveniently gives title/date/body directly, no HTML-parsing guesswork needed).

### Configuration

In [ ]:
jpost_sitemap_index = "https://www.jpost.com/jpgooglesitemap/sitemapindex.xml"

jpost_urls_file = "jerusalem_post_urls.csv"
jpost_output_file = "jerusalem_post_articles.csv"

# Remove sections that are never relevant to conflict coverage
jpost_remove_sections = [
    "travel", "us-elections", "shopping", "services", "videos", "german",
    "business-and-innovation", "astrology", "home-services", "health-and-wellness",
    "history", "conferences", "crypto-currency", "real-estate", "science", "advisor",
    "influencers", "magazine", "annual-conference", "spanish", "365days",
    "50-most-influential-jews",]

# Section actually scraped for full article text in step 3 below
jpost_target_section = "israel-news"

### Discover article URLs from the sitemap

In [ ]:
jpost_saved_urls = load_saved_keys(jpost_urls_file, key_col="url")

index_html = get_html(jpost_sitemap_index, timeout=60)
index_text = (index_html or "").lstrip("\ufeff")
all_sitemaps = re.findall(r"<loc>(.*?)</loc>", index_text)
article_sitemaps = [s for s in all_sitemaps if "SiteMap_Articles" in s]
print("Article sitemaps found:", len(article_sitemaps))

for sitemap_url in article_sitemaps:
    print("\nProcessing:", sitemap_url)
    try:
        response = requests.get(sitemap_url, headers=headers, timeout=120)
        if response.status_code != 200:
            print("Failed:", response.status_code)
            continue

        xml = gzip.decompress(response.content).decode("utf-8", errors="ignore")
        entries = re.findall(r"<url>(.*?)</url>", xml, flags=re.DOTALL)

        saved_here = 0
        for entry in entries:
            loc_match = re.search(r"<loc>(.*?)</loc>", entry)
            lastmod_match = re.search(r"<lastmod>(.*?)</lastmod>", entry)
            if not loc_match:
                continue

            url = loc_match.group(1).strip()
            if url in jpost_saved_urls:
                continue

            lastmod = lastmod_match.group(1).strip() if lastmod_match else None
            # rough string-compare date filter (ISO-formatted lastmod dates sort lexically)
            if lastmod and lastmod[:10] < start_date.strftime("%Y-%m-%d"):
                continue

            append_row({"url": url, "lastmod": lastmod}, jpost_urls_file)
            jpost_saved_urls.add(url)
            saved_here += 1

        print("Saved:", saved_here)
        time.sleep(random.uniform(3, 5))  # polite delay between sitemap files

    except Exception as e:
        print("Error:", e)

print("\nFinished discovery. Total URLs saved:", len(jpost_saved_urls))

### Filter to relevant sections

In [ ]:
jpost_urls_df = pd.read_csv(jpost_urls_file)
jpost_urls_df["section"] = (
    jpost_urls_df["url"].str.replace("https://www.jpost.com/", "", regex=False).str.split("/").str[0])

jpost_urls_df = jpost_urls_df[~jpost_urls_df["section"].isin(jpost_remove_sections)].reset_index(drop=True)
jpost_urls_df.to_csv(jpost_urls_file, index=False)

print(jpost_urls_df["section"].value_counts().head(30))

### Scrape article text (JSON-LD) for the target section

In [ ]:
def scrape_jpost_article(url):
    html = get_html(url)
    if html is None:
        return None
    try:
        soup = BeautifulSoup(html, "html.parser")

        # Find the NewsArticle JSON-LD block containing articleBody
        data = None
        for script in soup.find_all("script", type="application/ld+json"):
            try:
                parsed = json.loads(script.string)
            except Exception:
                continue
            if isinstance(parsed, list):
                for item in parsed:
                    if isinstance(item, dict) and "articleBody" in item:
                        data = item
                        break
            elif isinstance(parsed, dict) and "articleBody" in parsed:
                data = parsed
            if data:
                break

        if not data:
            return None

        date_str = data.get("datePublished")
        if not date_str:
            return None
        article_date = datetime.fromisoformat(date_str.replace("Z", "+00:00")).replace(tzinfo=None)
        if article_date < start_date:
            return None

        content = data.get("articleBody", "")
        if not content:
            return None

        matched_keywords = find_matching_keywords(content)
        if not matched_keywords:
            return None  # only keep articles matching the conflict keyword list

        section = data.get("articleSection", "")
        if isinstance(section, list):
            section = ", ".join(section)

        return {
            "url": url,
            "source": "Jerusalem Post",
            "title": data.get("headline"),
            "date": article_date.strftime("%d-%m-%Y"),
            "section": section,
            "content": content,
            "keywords": ", ".join(matched_keywords),
        }
    except Exception as e:
        print("Error:", url, e)
        return None


jpost_target_df = jpost_urls_df[jpost_urls_df["section"] == jpost_target_section].reset_index(drop=True)
jpost_saved_article_urls = load_saved_keys(jpost_output_file, key_col="url")

for url in tqdm(jpost_target_df["url"].tolist()):
    if url in jpost_saved_article_urls:
        continue

    article = scrape_jpost_article(url)
    if article is not None:
        append_row(article, jpost_output_file)
        jpost_saved_article_urls.add(url)

    time.sleep(request_delay)

print("Jerusalem Post scraping complete.")

---
## 5. Israel National News

[israelnationalnews.com](https://www.israelnationalnews.com) doesn't expose a sitemap or sequential article IDs, and its content is rendered client-side, so this section uses [Selenium](https://www.selenium.dev/) to drive a real browser instead of plain `requests`. It works in two steps: **Step 1** runs each keyword from the shared list through the site's search page and collects result links, and **Step 2** visits each collected link to pull the full article text and publish date, then filters to articles on/after the shared `start_date`.

This needs Google Chrome and a matching [ChromeDriver](https://chromedriver.chromium.org/) installed and on your `PATH` (see `README.md` for setup).

### Browser setup

In [ ]:
inn_driver = webdriver.Chrome()
inn_wait = WebDriverWait(inn_driver, 15)

### Configuration

In [ ]:
inn_base_url = "https://www.israelnationalnews.com"
inn_search_base_url = f"{inn_base_url}/search"

inn_results_csv = "israelnationalnews_raw.csv"     # search-result links, filled in during Step 1
inn_filtered_csv = "israelnationalnews_articles.csv"  # full article text, produced by Step 2

inn_resume_from_keyword_index = 0  # index into `keywords` to resume from (0 = start of list)
inn_resume_from_page = 1           # page number within that keyword to resume from
inn_resume_from_row_index = 0      # Step 2: row index in the CSV to resume from

### Helper functions

In [ ]:
def build_inn_search_url(keyword, page_n):
    # Google CSE deep-link format: #gsc.tab=0&gsc.q=QUERY&gsc.sort=date&gsc.page=N
    return f"{inn_search_base_url}#gsc.tab=0&gsc.q={quote(keyword)}&gsc.sort=date&gsc.page={page_n}"


def wait_for_inn_page(page_n, retries=3):
    for attempt in range(retries):
        try:
            # Results loaded
            inn_wait.until(lambda d:
                d.find_elements(By.CSS_SELECTOR, "a.gs-title") and
                d.find_element(By.CSS_SELECTOR, "a.gs-title").text.strip() != ""
            )
            # For pages beyond the first, confirm the active cursor page matches
            # (the cursor box only exists once there is more than one page of results)
            if page_n > 1:
                inn_wait.until(lambda d:
                    d.find_elements(By.CSS_SELECTOR, "div.gsc-cursor-current-page") and
                    d.find_element(By.CSS_SELECTOR, "div.gsc-cursor-current-page").text.strip() == str(page_n)
                )
            return  # success
        except TimeoutException:
            if attempt < retries - 1:
                print(f"  Timeout on page {page_n}, reloading (attempt {attempt + 1}/{retries})...")
                inn_driver.refresh()
                time.sleep(2 ** attempt)  # 1s -> 2s -> 4s
            else:
                raise


def scrape_inn_keyword(data, keyword, keyword_idx=0, start_page=1):
    # Go straight to start_page (page 1 on a normal run, or wherever we're resuming
    # from) — the cursor box shows the full page range regardless of which page
    # you're currently on, so there's no need to visit page 1 first just to count pages.
    inn_driver.get(build_inn_search_url(keyword, start_page))
    wait_for_inn_page(start_page)
    soup = BeautifulSoup(inn_driver.page_source, "html.parser")

    page_buttons = soup.select("div.gsc-cursor-page")
    if page_buttons:
        num_pages = max(
            int(b.get_text(strip=True)) for b in page_buttons if b.get_text(strip=True).isdigit()
        )
    else:
        num_pages = 1
    print(f"[keyword_idx={keyword_idx}] [{keyword}] {num_pages} pages found")

    if start_page > 1:
        print(f"[keyword_idx={keyword_idx}] [{keyword}] Resuming from page {start_page}")

    for page_n in range(start_page, num_pages + 1):
        print(f"[keyword_idx={keyword_idx}] [{keyword}] Scraping page {page_n}/{num_pages}  "
              f"(resume coords: inn_resume_from_keyword_index={keyword_idx}, inn_resume_from_page={page_n})")
        if page_n != start_page:
            # We already have start_page's soup loaded from above.
            inn_driver.get(build_inn_search_url(keyword, page_n))
            wait_for_inn_page(page_n)
            soup = BeautifulSoup(inn_driver.page_source, "html.parser")

        articles = []
        for result in soup.select("div.gsc-expansionArea > div.gsc-webResult.gsc-result"):
            title_tag = result.select_one("a.gs-title")
            href = title_tag["href"] if title_tag else None

            # Keep only real articles (path ends in a numeric article id),
            # this drops homepage / section-front results Google sometimes includes.
            if not href or not href.rstrip("/").rsplit("/", 1)[-1].isdigit():
                continue

            title = " ".join(title_tag.get_text(" ").split()) if title_tag else None

            breadcrumb_tag = result.select_one("div.gs-visibleUrl-breadcrumb")
            label = None
            if breadcrumb_tag:
                parts = breadcrumb_tag.get_text(" ", strip=True).split("›")
                label = parts[-1].strip() or None if parts else None

            if not title:
                continue

            # No publish date is shown on the search results page for this site;
            # it's filled in during Step 2 from the article page itself.
            articles.append({"keyword": keyword, "title": title, "label": label, "link": href, "date": None})

        data = pd.concat([data, pd.DataFrame(articles)], ignore_index=True)
        data.to_csv(inn_results_csv, index=False)

        time.sleep(random.uniform(1.0, 2.5))

    return data

### Collect search result links by keyword

In [ ]:
if os.path.exists(inn_results_csv):
    inn_data = pd.read_csv(inn_results_csv)
    print(f"Loaded {len(inn_data)} previously scraped rows from {inn_results_csv}")
else:
    inn_data = pd.DataFrame(columns=["keyword", "title", "label", "link", "date"])

for idx, keyword in enumerate(keywords):
    if idx < inn_resume_from_keyword_index:
        print(f"[keyword_idx={idx}] [{keyword}] Skipping (already done)")
        continue
    start_page = inn_resume_from_page if idx == inn_resume_from_keyword_index else 1
    inn_data = scrape_inn_keyword(inn_data, keyword, keyword_idx=idx, start_page=start_page)

inn_data.drop_duplicates(subset=["link"], inplace=True)
inn_data.to_csv(inn_results_csv, index=False)

### Fetch full article text and filter by date

In [ ]:
inn_data = pd.read_csv(inn_results_csv)
if "article" not in inn_data.columns:
    inn_data["article"] = None
# The 'date' column comes in as all-NaN/float64 from Step 1 (every row was written
# with date=None). Force it to object dtype so we can write actual datetime values
# into individual cells without pandas raising a LossySetitemError/TypeError on the
# dtype mismatch.
inn_data["date"] = inn_data["date"].astype(object)

for n, link in enumerate(inn_data.link):
    if n < inn_resume_from_row_index:
        continue

    print(f"Fetching article {n + 1}/{len(inn_data)}  (resume coords: inn_resume_from_row_index={n})")

    text = None
    article_date = None
    try:
        inn_driver.get(link)
        inn_wait.until(ec.presence_of_element_located((By.CSS_SELECTOR, "div#articleContent")))
        soup = BeautifulSoup(inn_driver.page_source, "html.parser")

        paragraphs = soup.select("div#articleContent p")
        text = " ".join(p.get_text(" ", strip=True) for p in paragraphs)

        date_span = soup.select_one("time.article-date-gregorian span")
        if date_span:
            m = re.match(r"([A-Za-z]{3} \d{1,2}, \d{4})", date_span.get_text(strip=True))
            if m:
                article_date = datetime.strptime(m.group(1), "%b %d, %Y")

    except Exception as e:
        print(f"  Failed: {link} — {e}")

    inn_data.at[n, "article"] = text
    inn_data.at[n, "date"] = article_date

    if n % 10 == 0:
        inn_data.to_csv(inn_results_csv, index=False)
    time.sleep(random.uniform(1.0, 2.5))

inn_data.to_csv(inn_results_csv, index=False)

### Standardize and filter

In [ ]:
inn_data["date"] = pd.to_datetime(inn_data["date"])
inn_filtered = inn_data[inn_data["date"] >= start_date].reset_index(drop=True)

inn_filtered = inn_filtered.rename(columns={"link": "url", "article": "content", "label": "section"})
inn_filtered["source"] = "Israel National News"
inn_filtered["keywords"] = inn_filtered["content"].apply(
    lambda text: ", ".join(find_matching_keywords(text)) if pd.notna(text) else "")

inn_filtered.to_csv(inn_filtered_csv, index=False, encoding="utf-8-sig")

print("Israel National News scraping complete:", len(inn_filtered), "articles")

### Close the browser

In [ ]:
inn_driver.quit()